WHY THIS EXPERIMENT: window=64 beat window=48 by only 0.007 in
validation loss a gap small enough to plausibly be random
variation from weight initialization, not a real advantage.
We retrain the top 3 candidates (16, 24, 64) three times each
with different random seeds, then compare MEAN and STD of their
validation loss. If one window's mean is clearly lower AND the
ranges don't overlap much, that's real evidence it's better.
If the ranges overlap heavily, the original "winner" was likely
just luck.

In [1]:

from google.colab import files
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [2]:
uploaded = files.upload()   # upload part1_outputs.zip

import zipfile
with zipfile.ZipFile('part1_outputs.zip', 'r') as z:
    z.extractall('.')
print("Files extracted.")

Saving part1_outputs.zip to part1_outputs.zip
Files extracted.


In [3]:
# Setup
CANDIDATE_WINDOWS = [16, 24, 64]
SEEDS = [42, 123, 2026]
EPOCHS = 100
BATCH_SIZE = 64
PATIENCE = 5   # confirmed correct by our earlier experiment


def load_window_data(window_size):
    X_train = np.load(f'X_train_w{window_size}.npy')
    y_train = np.load(f'y_train_w{window_size}.npy')
    X_val   = np.load(f'X_val_w{window_size}.npy')
    y_val   = np.load(f'y_val_w{window_size}.npy')
    return X_train, y_train, X_val, y_val


def build_lstm_model(window_size, n_features):
    model = keras.Sequential([
        layers.Input(shape=(window_size, n_features)),
        layers.LSTM(64, return_sequences=True),
        layers.Dropout(0.2),
        layers.LSTM(32),
        layers.Dropout(0.2),
        layers.Dense(1)
    ])
    return model


def train_with_seed(window_size, seed):
    # WHY set all three seeds: TensorFlow's randomness comes from
    # multiple sources (numpy, python's random, and TF's own graph
    # ops) — setting only one would leave the others uncontrolled.
    tf.random.set_seed(seed)
    np.random.seed(seed)

    X_train, y_train, X_val, y_val = load_window_data(window_size)
    n_features = X_train.shape[2]

    model = build_lstm_model(window_size, n_features)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)
    ]

    print(f"\n--- window={window_size}, seed={seed} ---")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=0   # quiet we only need the final numbers, not epoch-by-epoch logs
    )

    best_val_loss = min(history.history['val_loss'])
    print(f"    best_val_loss = {best_val_loss:.5f} (stopped at epoch {len(history.history['loss'])})")
    return best_val_loss


In [4]:
# Run: 3 windows x 3 seeds = 9 trainings
seed_results = {w: [] for w in CANDIDATE_WINDOWS}

for w in CANDIDATE_WINDOWS:
    for s in SEEDS:
        val_loss = train_with_seed(w, s)
        seed_results[w].append(val_loss)


--- window=16, seed=42 ---
    best_val_loss = 0.25806 (stopped at epoch 10)

--- window=16, seed=123 ---
    best_val_loss = 0.26255 (stopped at epoch 9)

--- window=16, seed=2026 ---
    best_val_loss = 0.25395 (stopped at epoch 8)

--- window=24, seed=42 ---
    best_val_loss = 0.26775 (stopped at epoch 7)

--- window=24, seed=123 ---
    best_val_loss = 0.26189 (stopped at epoch 7)

--- window=24, seed=2026 ---
    best_val_loss = 0.26142 (stopped at epoch 8)

--- window=64, seed=42 ---
    best_val_loss = 0.26671 (stopped at epoch 8)

--- window=64, seed=123 ---
    best_val_loss = 0.25843 (stopped at epoch 11)

--- window=64, seed=2026 ---
    best_val_loss = 0.25942 (stopped at epoch 9)


In [5]:
# Compare: mean, std, and overlap between candidates

print("\n=== Seed Repetition Results (mean ± std across 3 seeds) ===")
summary = {}
for w in CANDIDATE_WINDOWS:
    vals = seed_results[w]
    mean_v = np.mean(vals)
    std_v = np.std(vals)
    summary[w] = {'mean': mean_v, 'std': std_v, 'values': vals}
    print(f"Window {w}: {vals} -> mean={mean_v:.5f}, std={std_v:.5f}")

best_by_mean = min(summary, key=lambda w: summary[w]['mean'])
print(f"\nBest by mean validation loss: window={best_by_mean}")

# WHY check overlap: if window A's (mean - std) is still higher than
# window B's (mean + std), A is clearly, robustly worse -- not just
# by luck. If the ranges overlap, we can't confidently claim a winner.
print("\n=== Overlap check (is the difference statistically meaningful?) ===")
for w in CANDIDATE_WINDOWS:
    lower = summary[w]['mean'] - summary[w]['std']
    upper = summary[w]['mean'] + summary[w]['std']
    print(f"Window {w}: range = [{lower:.5f}, {upper:.5f}]")

best_range = (summary[best_by_mean]['mean'] - summary[best_by_mean]['std'],
              summary[best_by_mean]['mean'] + summary[best_by_mean]['std'])
overlaps = []
for w in CANDIDATE_WINDOWS:
    if w == best_by_mean:
        continue
    other_range = (summary[w]['mean'] - summary[w]['std'], summary[w]['mean'] + summary[w]['std'])
    overlap = not (best_range[1] < other_range[0] or other_range[1] < best_range[0])
    overlaps.append(overlap)
    print(f"Window {best_by_mean} vs Window {w}: ranges overlap = {overlap}")

if any(overlaps):
    print(f"\n=> CONCLUSION: The ranges overlap. Window {best_by_mean}'s advantage is "
          f"NOT statistically robust -- it could be random chance from initialization. "
          f"Report should state this honestly: performance is comparable across these window sizes.")
else:
    print(f"\n=> CONCLUSION: Window {best_by_mean} is robustly better across all 3 seeds, "
          f"with no overlap against the other candidates. This is genuine evidence, not luck.")



=== Seed Repetition Results (mean ± std across 3 seeds) ===
Window 16: [0.2580580711364746, 0.262554407119751, 0.25395315885543823] -> mean=0.25819, std=0.00351
Window 24: [0.2677466571331024, 0.26189154386520386, 0.2614220082759857] -> mean=0.26369, std=0.00288
Window 64: [0.2667093873023987, 0.2584323287010193, 0.25941693782806396] -> mean=0.26152, std=0.00369

Best by mean validation loss: window=16

=== Overlap check (is the difference statistically meaningful?) ===
Window 16: range = [0.25468, 0.26170]
Window 24: range = [0.26081, 0.26656]
Window 64: range = [0.25783, 0.26521]
Window 16 vs Window 24: ranges overlap = True
Window 16 vs Window 64: ranges overlap = True

=> CONCLUSION: The ranges overlap. Window 16's advantage is NOT statistically robust -- it could be random chance from initialization. Report should state this honestly: performance is comparable across these window sizes.


In [6]:
# Save results and download

import pandas as pd
rows = []
for w in CANDIDATE_WINDOWS:
    for i, s in enumerate(SEEDS):
        rows.append({'window': w, 'seed': s, 'val_loss': seed_results[w][i]})
df_results = pd.DataFrame(rows)
df_results.to_csv('seed_repetition_results.csv', index=False)
files.download('seed_repetition_results.csv')

print("\nExperiment complete. Results saved to seed_repetition_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Experiment complete. Results saved to seed_repetition_results.csv
